In [3]:
import numpy as np
import pandas as pd
import os
import glob


In [2]:
parquet_files = glob.glob("../data/processed/*.parquet")
merged = pd.read_parquet("../data/processed/loans_master.parquet")

merged.head()

,loan_id,issue_date,issue_year,issue_month,loan_amnt_inr,funded_amnt_inr,loan_term_months,int_rate_pct,installment_inr,annual_installment_inr,...,pymnt_plan,hardship_flag,initial_list_status,disbursement_method,verification_status,rbi_repo_rate_pct,gdp_growth_pct,cpi_inflation_pct,rate_spread_pct,real_interest_rate_pct
0,LN000000001,Feb-2016,2016,2,80678.0,74992.0,36,14.91,2793.17,33518.0,...,N,N,w,DIRECT_PAY,Source Verified,6.25,8.2,4.5,8.66,10.410000
1,LN000000002,May-2024,2024,5,274166.0,265041.0,36,7.00,8465.45,101585.0,...,N,N,w,CASH,Verified,6.50,6.8,4.9,0.50,2.100000
2,LN000000003,Dec-2021,2021,12,59603.0,54423.0,60,13.34,1366.55,16399.0,...,N,N,w,DIRECT_PAY,Source Verified,4.00,8.7,5.1,9.34,8.240000
3,LN000000004,Nov-2020,2020,11,246313.0,224181.0,84,24.07,6088.88,73067.0,...,N,N,w,DIRECT_PAY,Source Verified,4.00,-6.6,6.2,20.07,17.870001
4,LN000000005,Jul-2013,2013,7,101471.0,95361.0,60,8.52,2082.81,24994.0,...,Y,N,w,CASH,Verified,7.75,6.4,10.9,0.77,-2.380000


In [3]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Data columns (total 27 columns):
 #   Column                  Dtype   
---  ------                  -----   
 0   loan_id                 object  
 1   issue_date              category
 2   issue_year              int16   
 3   issue_month             int8    
 4   loan_amnt_inr           float32 
 5   funded_amnt_inr         float32 
 6   loan_term_months        int8    
 7   int_rate_pct            float32 
 8   installment_inr         float64 
 9   annual_installment_inr  float32 
 10  grade                   category
 11  sub_grade               category
 12  loan_purpose            category
 13  state_code              category
 14  region                  category
 15  urban_index             float32 
 16  application_type        category
 17  pymnt_plan              category
 18  hardship_flag           category
 19  initial_list_status     category
 20  disbursement_method     category
 21  verifica

In [ ]:
join_summary = []
for f in parquet_files:
    key=os.path.basename(f).replace(".parquet","")
    if key == "loans_master":
        continue
    df=pd.read_parquet(f)

    # For every loan_id in merged, does it exist in df?    
    orphan_count=(~merged["loan_id"].isin(df["loan_id"])).sum()
    
    merged = merged.merge(df,on="loan_id",how="left")

    join_summary.append({
        "table":key,
        "row_after_join":merged.shape[0],
        "orphans":orphan_count
    })
    print(f" + {key:35s} -> shape: {merged.shape}")
join_summary = pd.DataFrame(join_summary)
markdown_table = join_summary.to_markdown(index=False)
with open ("../report/figures/data_acqu_join_clean.md","a") as f:
    f.write("\n\n## Join Summary\n\n")
    f.write(markdown_table)


 + customer_bureau                     -> shape: (2000000, 56)
 + loan_performance                    -> shape: (2000000, 67)
 + payment_history                     -> shape: (2000000, 84)
 + branch_region_economy               -> shape: (2000000, 102)
 + monthly_emi_track                   -> shape: (2000000, 124)
 + loan_enquiry_bureau                 -> shape: (2000000, 147)
 + credit_card_behavior                -> shape: (2000000, 163)
 + collateral_assets                   -> shape: (2000000, 182)


### Saving the merged dataset

In [4]:
# merged.to_parquet("../data/merged/final_merged_dataset.parquet",index=False)
merged=pd.read_parquet("../data/merged/final_merged_dataset.parquet")

In [3]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Columns: 182 entries, loan_id to business_asset_val_inr
dtypes: category(42), float32(55), float64(29), int16(3), int32(1), int8(49), object(3)
memory usage: 1.1+ GB


In [4]:
merged.duplicated().sum()

np.int64(0)

In [4]:
missing_summary=merged.isna().sum()
missing_summary[missing_summary>0].sort_values(ascending=False)

vehicle_type                 1879789
property_city_tier           1679672
property_type                1679672
mths_since_last_record       1598332
ltv_ratio_pct                1472196
valuation_agency             1370048
collateral_type              1258896
charge_type                  1258896
mths_since_last_delinq       1098640
primary_cc_bank               560528
primary_card_type             560528
top_spend_category            560528
income_doc_type               308704
collateral_score              298379
mort_acc                      260330
il_util_pct                   200167
emp_length_years              180539
bc_util_pct                   160397
cash_advance_inr              160223
cc_payment_score              160149
branch_sanction_rate          140557
revol_util_pct                140097
loan_officer_exp_years        139366
avg_payment_delay_days        120364
emi_bank_name                 120236
collection_recovery_fee       120086
pdc_count                     120057
p

In [ ]:
# merged.describe(include="all").T.to_excel("../report/summaries/descriptive_stats.xlsx")

In [ ]:
# merged.describe().T.to_excel("../report/summaries/numerical_col_stats.xlsx")

### Dirty Flag

In [36]:
# Much percentage fields should satisfy 0 <= value <= 100
pct_cols=[c for c in merged.columns if "pct" in c]
# print(pct_cols)

for col in pct_cols:
    cnt=(
        (merged[col]<0) | (merged[col]>100)
    ).sum()

    if cnt>0:
        print(col,cnt)


gdp_growth_pct 139881
rate_spread_pct 23286
real_interest_rate_pct 76546
rejection_rate_pct 3168
ltv_ratio_pct 28700


Column
1. rejection_rate_pct-Negative rejection_rates make no sense
2. ltv (loan to value) ratio - negative almost certainly invalid

In [32]:
# Inr field should not be negative
# Chekcing INR fileds
inr_cols = [c for c in merged.columns if "inr" in c]

for col in inr_cols:
    cnt=((merged[col]<0)).sum()

    if cnt>0:
        print(col,cnt)

In [ ]:
# Assignments frequently inject [-999,999,9999,99999]
# numbers like -999, 999, 9999, and 99999 are used as sentinel values or missing data placeholders.
# sentinel value
num_cols = merged.select_dtypes(include="number").columns

col="col_name"
val="value"
cnt="count"

print(f"{col:25s}{val:15s}{cnt:15s}")

for col in num_cols:
    for val in [-999,999,9999,99999]:
        cnt=(merged[col]==val).sum()

        if cnt>0:
            print(f"{col:<25s}{val:<15d}{cnt:<15d}")


col_name                 value          count          
loan_amnt_inr            99999          11             
funded_amnt_inr          99999          6              
installment_inr          999            1              
annual_installment_inr   9999           1              
annual_installment_inr   99999          4              
revol_bal_inr            999            2              
revol_bal_inr            9999           2              
avg_cur_bal_inr          999            1              
provision_inr            999            9              
total_rec_prncp_inr      99999          11             
last_pymnt_amnt_inr      999            1              
last_pymnt_amnt_inr      9999           1              
expected_loss_inr        999            2              
installment_due_inr      999            1              
total_emi_due_inr        99999          3              
emi_overdue_inr          999            2              
emi_advance_paid_inr     999            1       

In [33]:
for col in [
    "property_area_sqft",
    "avg_monthly_cc_spend_inr",
    "cash_advance_inr"
]:
    print("\n", col)
    print(merged[col].value_counts().head(20))


 property_area_sqft
property_area_sqft
0       1679672
3456        105
4506        104
3396        103
4125        101
3985        100
2134         99
3535         99
764          98
4960         98
2499         98
514          97
4271         97
4888         96
4833         96
3075         96
1980         96
3511         96
2764         95
3972         95
Name: count, dtype: int64

 avg_monthly_cc_spend_inr
avg_monthly_cc_spend_inr
0.0       560528
2006.0       169
2471.0       160
2050.0       159
2478.0       157
2307.0       155
2360.0       153
2425.0       153
2753.0       152
1727.0       151
3517.0       151
1896.0       150
1656.0       150
2178.0       150
2218.0       149
1755.0       148
1893.0       148
1857.0       148
2368.0       148
1918.0       148
Name: count, dtype: int64

 cash_advance_inr
cash_advance_inr
0.0       1442038
537.0         118
551.0         115
664.0         113
944.0         112
708.0         111
919.0         111
1052.0        110
758.0         10

#### Column:
Most houses are 1200-2500 sqft
3. property_area_sqft - 999 need to inspect,property_area_sqft=0 for 1679672 alomst 84% value ?
4. avg_monthly _cc_spend_inr - 999, 86 records/9999-55 records,pattern ? avg_monthly_cc_spend_inr = 0 ?
5. cash_advance_inr - 999

In [39]:
obj_cols = merged.select_dtypes(include="object").columns

for col in obj_cols:
    print("\n", col)
    print(
        merged[col]
        .value_counts(dropna=False)
        .head(20)
    )


 loan_id
loan_id
LN000000001    1
LN000000002    1
LN000000003    1
LN000000004    1
LN000000005    1
LN000000006    1
LN000000007    1
LN000000008    1
LN000000009    1
LN000000010    1
LN000000011    1
LN000000012    1
LN000000013    1
LN000000014    1
LN000000015    1
LN000000016    1
LN000000017    1
LN000000018    1
LN000000019    1
LN000000020    1
Name: count, dtype: int64

 customer_id
customer_id
CU00411848    11
CU00289120    11
CU00565558    11
CU00460861    10
CU00775300    10
CU01265949    10
CU00603505    10
CU00466167    10
CU00036485    10
CU01282995     9
CU00782633     9
CU00338986     9
CU00963347     9
CU00074559     9
CU00524510     9
CU00212973     9
CU01143056     9
CU00812855     9
CU00104995     9
CU00243239     9
Name: count, dtype: int64

 branch_id
branch_id
BR7328-MH    52
BR6972-MH    49
BR6091-MH    48
BR1331-MH    48
BR6428-MH    48
BR9906-MH    47
BR9647-MH    47
BR5156-MH    46
BR8391-MH    46
BR8891-MH    46
BR6435-MH    46
BR1728-MH    46
BR8595-MH 

In [46]:
print("emi_to_income_ration")
print(merged["emi_to_income_ratio"].describe())
print("Values > 1:", (merged["emi_to_income_ratio"] > 1).sum())
print("Values < 0:", (merged["emi_to_income_ratio"] < 0).sum())

emi_to_income_ration
count    2.000000e+06
mean     2.127907e-01
std      3.274482e-01
min      5.000000e-04
25%      5.200000e-02
50%      1.129000e-01
75%      2.430000e-01
max      2.858450e+01
Name: emi_to_income_ratio, dtype: float64
Values > 1: 52798
Values < 0: 0


###
6. emit_to_income_ration = max=28.58 - No one pays 28 X income as EMI

In [29]:
print("\n" + "="*60)
print("AGE vs EMP_LENGTH CONSISTENCY")
print("="*60)
# Person aged 22 cannot have 15 years employment
impossible_emp = merged["emp_length_years"] > (merged["age"] - 18)
print(f"emp_length > (age - 18): {impossible_emp.sum()}")


AGE vs EMP_LENGTH CONSISTENCY
emp_length > (age - 18): 191754


In [30]:
merged['dirty_flag']=0

In [31]:
merged.head()

,loan_id,issue_date,issue_year,issue_month,loan_amnt_inr,funded_amnt_inr,loan_term_months,int_rate_pct,installment_inr,annual_installment_inr,...,valuation_date,valuation_agency,charge_type,collateral_score,prop_value_inr,vehicle_value_inr,business_asset_val_inr,mths_since_last_delinq_missing,il_util_pct_missing,dirty_flag
0,LN000000001,Feb-2016,2016,2,80678.0,74992.0,36,14.91,2793.17,33518.0,...,May-2016,NaN,NaN,0.000000,0.0,0.0,0.0,0,0,0
1,LN000000002,May-2024,2024,5,274166.0,265041.0,36,7.00,8465.45,101585.0,...,Jul-2023,JLL,First Charge,72.000000,0.0,0.0,607261.0,0,0,0
2,LN000000003,Dec-2021,2021,12,59603.0,54423.0,60,13.34,1366.55,16399.0,...,Oct-2021,Cushman,Equitable Mortgage,48.099998,4194101.0,0.0,0.0,0,0,0
3,LN000000004,Nov-2020,2020,11,246313.0,224181.0,84,24.07,6088.88,73067.0,...,Mar-2020,ICRA,First Charge,NaN,3462228.0,0.0,0.0,0,0,0
4,LN000000005,Jul-2013,2013,7,101471.0,95361.0,60,8.52,2082.81,24994.0,...,Feb-2012,NaN,NaN,0.000000,0.0,0.0,0.0,0,0,0


In [35]:
merged['property_area_sqft'].eq(0).mean()

np.float64(0.839836)

In [37]:
# before labeling it dirty, check whether it appears unnaturally:
vc = merged['property_area_sqft'].value_counts().sort_index()

vc.loc[950:1050]

property_area_sqft
950     69
951     74
952     76
953     66
954     84
        ..
1046    78
1047    73
1048    69
1049    68
1050    72
Name: count, Length: 101, dtype: int64

In [46]:
merged['emi_to_income_ratio'].quantile(
    [0.90, 0.95, 0.99, 0.995, 0.999, 0.9999]
)

0.9000    0.486300
0.9500    0.725800
0.9900    1.529700
0.9950    1.997600
0.9990    3.441201
0.9999    6.627801
Name: emi_to_income_ratio, dtype: float64

In [52]:
for col in merged.select_dtypes(include='number'):
    vc = merged[col].value_counts()

    rare = vc[vc <= 20]

    if len(rare):
        print(col)
        print(rare.head())

loan_amnt_inr
loan_amnt_inr
55690.0    20
62914.0    20
73267.0    20
57922.0    20
58642.0    20
Name: count, dtype: int64
funded_amnt_inr
funded_amnt_inr
69560.0    20
63132.0    20
73433.0    20
60004.0    20
91291.0    20
Name: count, dtype: int64
int_rate_pct
int_rate_pct
26.559999    20
26.740000    20
27.170000    20
26.770000    20
26.700001    19
Name: count, dtype: int64
installment_inr
installment_inr
2457.96    20
1388.35    20
2397.81    20
2496.93    20
1510.62    20
Name: count, dtype: int64
annual_installment_inr
annual_installment_inr
30203.0    20
18465.0    20
41566.0    20
53148.0    20
51963.0    20
Name: count, dtype: int64
rate_spread_pct
rate_spread_pct
21.270000    20
21.240000    20
21.230000    20
21.330000    20
21.389999    19
Name: count, dtype: int64
real_interest_rate_pct
real_interest_rate_pct
21.870001    20
21.969999    20
22.410000    20
22.170000    20
21.910000    20
Name: count, dtype: int64
annual_inc_inr
annual_inc_inr
149869.0    15
158967.0   

In [59]:
dirty_mask = (
    (merged['rejection_rate_pct'] < 0) |
    (merged['ltv_ratio_pct'] < 0) |
    (merged['avg_monthly_cc_spend_inr'] == 999) |
    (merged['avg_monthly_cc_spend_inr'] == 9999) |
    (merged['cash_advance_inr'] == 999) |
    (merged['cash_advance_inr'] == 9999) |
    (merged['emp_length_years'] > (merged['age'] - 18)) |
    (merged['emi_to_income_ratio'] > 5)
)

merged['dirty_flag'] = dirty_mask.astype(int)

print("Dirty rows:", merged['dirty_flag'].sum())

Dirty rows: 192462


In [61]:
issues = {
    "Negative rejection_rate_pct":
        (merged['rejection_rate_pct'] < 0),

    "Negative ltv_ratio_pct":
        (merged['ltv_ratio_pct'] < 0),

    "avg_monthly_cc_spend_inr = 999":
        (merged['avg_monthly_cc_spend_inr'] == 999),

    "avg_monthly_cc_spend_inr = 9999":
        (merged['avg_monthly_cc_spend_inr'] == 9999),

    "cash_advance_inr = 999":
        (merged['cash_advance_inr'] == 999),

    "cash_advance_inr = 9999":
        (merged['cash_advance_inr'] == 9999),

    "emp_length_years > age-18":
        (merged['emp_length_years'] > (merged['age'] - 18)),

    "emi_to_income_ratio > 5":
        (merged['emi_to_income_ratio'] > 5)
}

for issue, mask in issues.items():
    print(f"{issue:35s} : {mask.sum():,}")

Negative rejection_rate_pct         : 0
Negative ltv_ratio_pct              : 0
avg_monthly_cc_spend_inr = 999      : 86
avg_monthly_cc_spend_inr = 9999     : 55
cash_advance_inr = 999              : 75
cash_advance_inr = 9999             : 5
emp_length_years > age-18           : 191,754
emi_to_income_ratio > 5             : 557


In [62]:
summary = pd.DataFrame({
    "Issue": issues.keys(),
    "Dirty_Record_Count": [mask.sum() for mask in issues.values()]
})

summary

,Issue,Dirty_Record_Count
0,Negative rejection_rate_pct,0
1,Negative ltv_ratio_pct,0
2,avg_monthly_cc_spend_inr = 999,86
3,avg_monthly_cc_spend_inr = 9999,55
4,cash_advance_inr = 999,75
5,cash_advance_inr = 9999,5
6,emp_length_years > age-18,191754
7,emi_to_income_ratio > 5,557


In [58]:
merged.loc[merged['dirty_flag'] == 1].shape

(192264, 185)

### Missing Value

In [43]:
miss_col=missing_summary[missing_summary>0].sort_values(ascending=False)
miss_col

vehicle_type                 1879789
property_city_tier           1679672
property_type                1679672
mths_since_last_record       1598332
ltv_ratio_pct                1472196
valuation_agency             1370048
collateral_type              1258896
charge_type                  1258896
mths_since_last_delinq       1098640
primary_cc_bank               560528
primary_card_type             560528
top_spend_category            560528
income_doc_type               308704
collateral_score              298379
mort_acc                      260330
il_util_pct                   200167
emp_length_years              180539
bc_util_pct                   160397
cash_advance_inr              160223
cc_payment_score              160149
branch_sanction_rate          140557
revol_util_pct                140097
loan_officer_exp_years        139366
avg_payment_delay_days        120364
emi_bank_name                 120236
collection_recovery_fee       120086
pdc_count                     120057
p

In [19]:
len(miss_col)

38

Categories used to classify missing data.
1. MCAR - Missing completely at Random // Missingness has no relationship with any variable.
2. MAR - Missing at Random // Missingness depends on other observed variables.
3. MNAR - Missing Not at Random // Missingness depends on the value itself.


In [20]:
missing_cols=['mths_since_last_delinq','mort_acc','emp_length_years','il_util_pct']
merged[missing_cols].isnull().sum()

mths_since_last_delinq    0
mort_acc                  0
emp_length_years          0
il_util_pct               0
dtype: int64

In [21]:
merged['mths_since_last_delinq_missing']=(merged['mths_since_last_delinq'].isnull().astype(int))
merged['mths_since_last_delinq']=(merged['mths_since_last_delinq'].fillna(-1))
merged.head()

,loan_id,issue_date,issue_year,issue_month,loan_amnt_inr,funded_amnt_inr,loan_term_months,int_rate_pct,installment_inr,annual_installment_inr,...,insurance_flag,valuation_date,valuation_agency,charge_type,collateral_score,prop_value_inr,vehicle_value_inr,business_asset_val_inr,mths_since_last_delinq_missing,il_util_pct_missing
0,LN000000001,Feb-2016,2016,2,80678.0,74992.0,36,14.91,2793.17,33518.0,...,0,May-2016,NaN,NaN,0.000000,0.0,0.0,0.0,0,0
1,LN000000002,May-2024,2024,5,274166.0,265041.0,36,7.00,8465.45,101585.0,...,1,Jul-2023,JLL,First Charge,72.000000,0.0,0.0,607261.0,0,0
2,LN000000003,Dec-2021,2021,12,59603.0,54423.0,60,13.34,1366.55,16399.0,...,0,Oct-2021,Cushman,Equitable Mortgage,48.099998,4194101.0,0.0,0.0,0,0
3,LN000000004,Nov-2020,2020,11,246313.0,224181.0,84,24.07,6088.88,73067.0,...,1,Mar-2020,ICRA,First Charge,NaN,3462228.0,0.0,0.0,0,0
4,LN000000005,Jul-2013,2013,7,101471.0,95361.0,60,8.52,2082.81,24994.0,...,0,Feb-2012,NaN,NaN,0.000000,0.0,0.0,0.0,0,0


In [22]:
merged['il_util_pct_missing'] = (
    merged['il_util_pct'].isnull().astype(int)
)

merged['il_util_pct'] = (
    merged['il_util_pct'].fillna(0)
)

In [23]:
merged['mort_acc'] = (
    merged['mort_acc']
    .fillna(merged['mort_acc'].median())
)

In [24]:
merged['emp_length_years'] = (
    merged['emp_length_years']
    .fillna(merged['emp_length_years'].median())
)

In [26]:
print("After Imputation")
print(merged[missing_cols].isnull().sum())

After Imputation
mths_since_last_delinq    0
mort_acc                  0
emp_length_years          0
il_util_pct               0
dtype: int64


### Winsoriation 

In [5]:
num_cols=merged.select_dtypes(include=np.number).columns
num_cols

Index(['issue_year', 'issue_month', 'loan_amnt_inr', 'funded_amnt_inr',
       'loan_term_months', 'int_rate_pct', 'installment_inr',
       'annual_installment_inr', 'urban_index', 'rbi_repo_rate_pct',
       ...
       'ltv_ratio_pct', 'loan_secured_flag', 'property_age_years',
       'property_area_sqft', 'vehicle_age_years', 'insurance_flag',
       'collateral_score', 'prop_value_inr', 'vehicle_value_inr',
       'business_asset_val_inr'],
      dtype='object', length=137)

In [6]:
skew_df= (merged[num_cols].skew().abs().sort_values(ascending=False))
top6_skewed = skew_df.head(6)
print(top6_skewed)

npa_flag                   365.144538
collections_12mths_fee     139.035412
collection_recovery_fee    125.765963
recoveries_inr              94.232480
emi_advance_paid_inr        59.171673
expected_loss_inr           27.014741
dtype: float64


In [7]:
before_stats = pd.DataFrame({
    'mean': merged[top6_skewed.index].mean(),
    'std': merged[top6_skewed.index].std(),
    'max': merged[top6_skewed.index].max()
})

before_stats.columns = [
    'mean_before',
    'std_before',
    'max_before'
]

In [8]:

winsor_df = merged.copy() # creating a copy so that original dataset remains unchanged

for col in top6_skewed.index:

    lower = winsor_df[col].quantile(0.01)
    upper = winsor_df[col].quantile(0.99)

    winsor_df[col] = winsor_df[col].clip(
        lower=lower,
        upper=upper
    )

In [9]:
after_stats = pd.DataFrame({
    'mean_after': winsor_df[top6_skewed.index].mean(),
    'std_after': winsor_df[top6_skewed.index].std(),
    'max_after': winsor_df[top6_skewed.index].max()
})

In [10]:
winsor_summary = (
    before_stats
    .merge(
        after_stats,
        left_index=True,
        right_index=True
    )
    .reset_index()
    .rename(columns={'index':'column'})
)

winsor_summary

,column,mean_before,std_before,max_before,mean_after,std_after,max_after
0,npa_flag,0.000008,0.002739,1.00,0.000000,0.000000,0.0000
1,collections_12mths_fee,32.366464,560.070667,312618.52,11.829477,77.296128,657.2202
2,collection_recovery_fee,130.709817,2061.435988,1035421.73,55.189725,348.528316,2911.0688
3,recoveries_inr,52.749164,818.402452,313742.97,22.139089,140.021787,1172.0600
4,emi_advance_paid_inr,2204.312963,13599.895684,5329334.00,1708.033934,5752.164359,40329.1235
5,expected_loss_inr,139.237678,1285.912722,189646.84,84.350827,500.602668,3990.4218


In [12]:
len(winsor_df.columns)

182

In [ ]:
# winsor_df.to_parquet("../data/merged/winsor_dataset.parquet",index=False)